In [ ]:
!pip install pytigergraph chromadb sentence-transformers groq google-generativeai bert-score tiktoken requests streamlit pyngrok pyvis -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.0/351.0 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 128.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69

In [ ]:
!git clone https://github.com/tigergraph/graphrag.git

Cloning into 'graphrag'...
remote: Enumerating objects: 4517, done.
remote: Counting objects: 100% (906/906), done.
remote: Compressing objects: 100% (247/247), done.
remote: Total 4517 (delta 726), reused 659 (delta 659), pack-reused 3611 (from 1)
Receiving objects: 100% (4517/4517), 24.26 MiB | 28.96 MiB/s, done.
Resolving deltas: 100% (3430/3430), done.


In [ ]:
import os
os.chdir('graphrag')

In [ ]:
!ls

CHANGELOG.md  docker-compose.yml  graphrag-k8s.yml  README.md
chat-history  docs		  graphrag-ui	    report-service
common	      ecc		  LICENSE	    tests
configs       graphrag		  licenses	    VERSION


In [ ]:
from groq import Groq
client = Groq(api_key=GROQ_API_KEY)
resp = client.chat.completions.create(
  model='llama-3.3-70b-versatile',
  messages=[{'role':'user','content':'Say hello'}]
)
print(resp.choices[0].message.content)
print('Tokens used:', resp.usage.total_tokens)

Hello. How can I assist you today?
Tokens used: 47


In [ ]:
import requests, json, time
import tiktoken

enc = tiktoken.get_encoding('cl100k_base')

MEDICAL_TOPICS = [
    "Diabetes mellitus", "Hypertension", "Alzheimer disease", "Aspirin", "COVID-19",
    "Insulin", "Cancer", "Chemotherapy", "Antibiotic resistance", "Heart failure",
    "Stroke", "Parkinson disease", "HIV/AIDS", "Metformin", "Inflammation",
    "Cholesterol", "Vaccine", "Rheumatoid arthritis", "Depression", "Anxiety disorder",
    "Tuberculosis", "Malaria", "Dengue fever", "Pneumonia", "Hepatitis",
    "Kidney disease", "Liver disease", "Asthma", "Epilepsy", "Schizophrenia",
    "Migraine", "Psoriasis", "Eczema", "Lupus", "Multiple sclerosis",
    "Cystic fibrosis", "Sickle cell disease", "Hemophilia", "Anemia", "Leukemia",
    "Lymphoma", "Melanoma", "Breast cancer", "Prostate cancer", "Lung cancer",
    "Colorectal cancer", "Ovarian cancer", "Pancreatic cancer", "Glioblastoma",
    "Thyroid cancer", "Bladder cancer", "Cervical cancer", "Testicular cancer",
    "Endometriosis", "Polycystic ovary syndrome", "Menopause", "Osteoporosis",
    "Osteoarthritis", "Gout", "Back pain", "Carpal tunnel syndrome",
    "Irritable bowel syndrome", "Crohn's disease", "Ulcerative colitis",
    "Peptic ulcer", "Gastroesophageal reflux disease", "Coeliac disease",
    "Food allergy", "Lactose intolerance", "Pancreatitis", "Gallstones",
    "Hepatitis C", "Hepatitis B", "Cirrhosis", "Fatty liver disease",
    "Kidney stones", "Urinary tract infection", "Chronic kidney disease",
    "Acute kidney injury", "Glomerulonephritis", "Pyelonephritis",
    "Deep vein thrombosis", "Pulmonary embolism", "Peripheral artery disease",
    "Aneurysm", "Varicose veins", "Arrhythmia", "Atrial fibrillation",
    "Myocardial infarction", "Coronary artery disease", "Angina",
    "Cardiomyopathy", "Congenital heart defect", "Pericarditis",
    "Hypothyroidism", "Hyperthyroidism", "Hashimoto's thyroiditis",
    "Graves' disease", "Cushing's syndrome", "Addison's disease",
    "Type 1 diabetes", "Type 2 diabetes", "Gestational diabetes",
    "Hypoglycemia", "Hyperglycemia", "Diabetic neuropathy", "Diabetic retinopathy",
    "Obesity", "Anorexia nervosa", "Bulimia nervosa", "Binge eating disorder",
    "Vitamin D deficiency", "Vitamin B12 deficiency", "Iron deficiency",
    "Zinc deficiency", "Magnesium deficiency", "Potassium imbalance",
    "Influenza", "Common cold", "Bronchitis", "Pneumothorax",
    "Sleep apnea", "Chronic obstructive pulmonary disease", "Emphysema",
    "Cystic fibrosis", "Pulmonary fibrosis", "Sarcoidosis",
    "Meningitis", "Encephalitis", "Sepsis", "Septic shock",
    "Lyme disease", "West Nile virus", "Zika virus", "Ebola virus disease",
    "Marburg virus", "Lassa fever", "Yellow fever", "Rabies",
    "Tetanus", "Diphtheria", "Pertussis", "Measles", "Mumps",
    "Rubella", "Chickenpox", "Shingles", "Herpes simplex",
    "Human papillomavirus", "Molluscum contagiosum",
    "Syphilis", "Gonorrhea", "Chlamydia", "Trichomoniasis",
    "Bacterial vaginosis", "Yeast infection", "Urinary incontinence",
    "Benign prostatic hyperplasia", "Prostatitis", "Erectile dysfunction",
    "Peyronie's disease", "Male infertility", "Female infertility",
    "Polycystic kidney disease", "Alport syndrome", "Bartter syndrome",
    "Gitelman syndrome", "Liddle's syndrome",
    "Osteogenesis imperfecta", "Marfan syndrome", "Ehlers-Danlos syndrome",
    "Achondroplasia", "Huntington's disease", "Amyotrophic lateral sclerosis",
    "Friedreich's ataxia", "Spinal muscular atrophy", "Duchenne muscular dystrophy",
    "Becker muscular dystrophy", "Myasthenia gravis",
    "Pituitary adenoma", "Acromegaly", "Prolactinoma",
    "Meniere's disease", "Tinnitus", "Vertigo", "Otitis media",
    "Conjunctivitis", "Glaucoma", "Cataract", "Macular degeneration",
    "Retinal detachment", "Diabetic retinopathy", "Color blindness",
    "Scurvy", "Rickets", "Pellagra", "Beriberi", "Keshan disease",
    "Acne", "Rosacea", "Vitiligo", "Alopecia areata", "Hirsutism",
    "Hyperhidrosis", "Nail fungus", "Athlete's foot", "Ringworm",
    "Scabies", "Lice infestation", "Bed bug infestation",
    "Attention deficit hyperactivity disorder", "Autism spectrum",
    "Bipolar disorder", "Obsessive-compulsive disorder",
    "Post-traumatic stress disorder", "Generalized anxiety disorder",
    "Panic disorder", "Social anxiety disorder", "Agoraphobia",
    "Specific phobia", "Dissociative identity disorder",
    "Somatic symptom disorder", "Conversion disorder",
    "Factitious disorder", "Munchausen syndrome by proxy",
    "Narcolepsy", "Insomnia", "Restless legs syndrome",
    "Circadian rhythm sleep disorder", "Sleepwalking",
    "Night terror", "Sleep paralysis",
    "Chronic fatigue syndrome", "Fibromyalgia",
    "Complex regional pain syndrome", "Temporomandibular joint disorder",
    "Interstitial cystitis", "Chronic prostatitis/chronic pelvic pain syndrome",
    "Vulvodynia", "Chronic Lyme disease",
    "Gulf War syndrome", "Sick building syndrome",
    "Multiple chemical sensitivity",
    "Altitude sickness", "Decompression sickness",
    "Heat stroke", "Hypothermia", "Frostbite",
    "Burns", "Wound healing", "Scar", "Keloid",
    "Skin cancer", "Basal cell carcinoma", "Squamous cell carcinoma",
    "Kaposi's sarcoma", "Merkel cell carcinoma",
    "Neuroblastoma", "Retinoblastoma", "Wilms tumor",
    "Hepatoblastoma", "Rhabdomyosarcoma", "Ewing sarcoma",
    "Osteosarcoma", "Chondrosarcoma", "Liposarcoma",
    "Synovial sarcoma", "Gastrointestinal stromal tumor",
    "Neuroendocrine tumor", "Carcinoid tumor", "Pheochromocytoma",
    "Paraganglioma", "Medullary thyroid cancer",
    "Adrenocortical carcinoma", "Thymoma", "Mesothelioma",
    "Pleural mesothelioma", "Peritoneal mesothelioma",
    "Adenocarcinoma", "Squamous cell carcinoma", "Small cell carcinoma",
    "Large cell carcinoma", "Non-small cell lung carcinoma",
    "Mesenchymal chondrosarcoma", "Desmoplastic small round cell tumor",
    "Primitive neuroectodermal tumor", "Atypical teratoid rhabdoid tumor",
    "Ibuprofen", "Paracetamol", "Acetaminophen", "Amoxicillin", "Azithromycin",
    "Omeprazole", "Lisinopril", "Amlodipine", "Simvastatin", "Atorvastatin",
    "Levothyroxine", "Metoprolol", "Albuterol", "Sertraline", "Fluoxetine",
    "Citalopram", "Escitalopram", "Venlafaxine", "Duloxetine", "Bupropion",
    "Furosemide", "Spironolactone", "Losartan", "Valsartan", "Clopidogrel",
    "Warfarin", "Apixaban", "Rivaroxaban", "Heparin", "Enoxaparin",
    "Morphine", "Oxycodone", "Codeine", "Tramadol", "Gabapentin",
    "Pregabalin", "Amitriptyline", "Nortriptyline", "Ciprofloxacin",
    "Levofloxacin", "Doxycycline", "Azithromycin", "Clarithromycin",
    "Metronidazole", "Fluconazole", "Ketoconazole", "Acyclovir",
    "Valacyclovir", "Oseltamivir", "Remdesivir", "Hydroxychloroquine",
    "Chloroquine", "Prednisone", "Prednisolone", "Dexamethasone",
    "Methylprednisolone", "Methotrexate", "Cyclophosphamide",
    "Tacrolimus", "Cyclosporine", "Mycophenolate",
    "Rituximab", "Adalimumab", "Infliximab", "Etanercept",
    "Insulin glargine", "Insulin aspart", "Glipizide", "Glyburide",
    "Sitagliptin", "Empagliflozin", "Dapagliflozin", "Liraglutide",
    "Semaglutide", "Orlistat", "Phentermine",
]
extra_topics = [
    "World Health Organization", "Public health", "Pharmacology",
    "Clinical trial", "Medical ethics", "Epidemiology",
    "Pathophysiology", "Genetic disorder", "Immune system",
    "Lymphatic system", "Nervous system", "Cardiovascular system",
    "Respiratory system", "Digestive system", "Endocrine system",
    "Reproductive system", "Integumentary system",
    "Medical research", "Evidence-based medicine",
    "Preventive healthcare", "Health policy",
    "Maternal health", "Infant mortality",
    "Antibiotic", "Antiviral drug", "Antifungal drug",
    "Anesthesia", "Surgery", "Radiation therapy"
]
MEDICAL_TOPICS.extend(extra_topics)
# Remove duplicates to avoid re‑scraping
MEDICAL_TOPICS = list(set(MEDICAL_TOPICS))

def get_wiki(title):
    url = 'https://en.wikipedia.org/w/api.php'
    headers = {
        'User-Agent': 'HackathonBot/1.0 (sachithags@gmail.com)'  # your email or a project name
    }
    params = {
        'action': 'query',
        'titles': title,
        'prop': 'extracts',
        'explaintext': True,
        'format': 'json'
    }
    try:
        r = requests.get(url, headers=headers, params=params)
        r.raise_for_status()  # raise error for non-200
        data = r.json()
        pages = data['query']['pages']
        for p in pages.values():
            return {'title': p.get('title',''), 'text': p.get('extract','')}
        return None
    except Exception as e:
        print(f"Failed to fetch {title}: {e}")
        return None

articles = []
for t in MEDICAL_TOPICS:
    a = get_wiki(t)
    if a and len(a['text']) > 500:
        articles.append(a)
    time.sleep(0.5)  # be polite to Wikipedia

total_tokens = sum(len(enc.encode(a['text'])) for a in articles)
print(f'Articles: {len(articles)}, Tokens: {total_tokens:,}')

Articles: 337, Tokens: 2,116,332


In [ ]:
collected = {a['title'] for a in articles}
for t in MEDICAL_TOPICS:
    if t in collected:
        continue
    a = get_wiki(t)
    if a and len(a['text']) > 500:
        articles.append(a)
        collected.add(t)
    time.sleep(0.5)

total_tokens = sum(len(enc.encode(a['text'])) for a in articles)
print(f'Articles: {len(articles)}, Tokens: {total_tokens:,}')

Articles: 337, Tokens: 2,116,332


In [ ]:
with open('medical_dataset.jsonl','w') as f:
    for a in articles:
        f.write(json.dumps(a)+'\n')
print('Saved', len(articles), 'articles')

Saved 337 articles


In [ ]:
BENCHMARK_QUESTIONS = [
  # Multi-hop (GraphRAG wins big here)
  'What drugs are used to treat inflammation in rheumatoid arthritis?',
  'How does insulin resistance lead to type 2 diabetes complications?',
  'What is the relationship between hypertension and kidney disease?',
  'Which antibiotics treat tuberculosis and what are their side effects?',
  'How does COVID-19 affect patients with pre-existing heart failure?',
  # Factual (all pipelines similar)
  'What is the mechanism of action of aspirin?',
  'What are the symptoms of Parkinson disease?',
  'How is malaria transmitted?',
  'What is the first line treatment for hypertension?',
  'What causes Alzheimer disease?'
  'Which diabetes drug interacts with aspirin and what are the potential risks?',
  'What chemotherapy agents are used for breast cancer and what are their common side effects?',
  'How does metformin help with polycystic ovary syndrome and what is the link to insulin resistance?',
  'Which vaccines are recommended for patients with asplenia and why?'
]
print('Questions ready:', len(BENCHMARK_QUESTIONS))

Questions ready: 13


In [ ]:
import time

def pipeline_llm_only(question):
    start = time.time()
    resp = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[
            {'role':'system','content':'Answer concisely.'},
            {'role':'user','content':question}
        ],
        max_tokens=500
    )
    latency = time.time() - start
    return {
        'answer': resp.choices[0].message.content,
        'tokens': resp.usage.total_tokens,
        'latency': round(latency, 2),
        'cost': resp.usage.total_tokens * 0.00000059  # Groq pricing
    }

# Test it
result = pipeline_llm_only('What is aspirin?')
print(result)

{'answer': "Aspirin is a pain reliever and anti-inflammatory medication, also known as acetylsalicylic acid (ASA). It's used to treat headaches, fever, and various types of pain.", 'tokens': 87, 'latency': 0.37, 'cost': 5.133e-05}


In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

# Load embedding model (small, fast on CPU)
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Create ChromaDB
chroma = chromadb.Client()
collection = chroma.create_collection('medical_rag')

# Chunk articles into 512-token pieces
def chunk_text(text, size=512, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), size-overlap):
        chunks.append(' '.join(words[i:i+size]))
    return chunks

# Index all articles (takes 10-20 min)
for i, article in enumerate(articles):
    chunks = chunk_text(article['text'])
    for j, chunk in enumerate(chunks):
        collection.add(
            documents=[chunk],
            ids=[f'doc_{i}_chunk_{j}'],
            metadatas=[{'title': article['title']}]
        )
    if i % 10 == 0: print(f'Indexed {i}/{len(articles)} articles')

def pipeline_basic_rag(question):
    start = time.time()
    results = collection.query(query_texts=[question], n_results=3)
    context = '\n\n'.join(results['documents'][0])
    resp = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[
            {'role':'system','content':'Answer using only this context: '+context},
            {'role':'user','content':question}
        ],
        max_tokens=500
    )
    latency = time.time() - start
    return {
        'answer': resp.choices[0].message.content,
        'tokens': resp.usage.total_tokens,
        'latency': round(latency, 2),
        'cost': resp.usage.total_tokens * 0.00000059
    }

result = pipeline_basic_rag('What drugs treat inflammation?')
print(result)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 34.2MiB/s]


Indexed 0/337 articles
Indexed 10/337 articles
Indexed 20/337 articles
Indexed 30/337 articles
Indexed 40/337 articles
Indexed 50/337 articles
Indexed 60/337 articles
Indexed 70/337 articles
Indexed 80/337 articles
Indexed 90/337 articles
Indexed 100/337 articles
Indexed 110/337 articles
Indexed 120/337 articles
Indexed 130/337 articles
Indexed 140/337 articles
Indexed 150/337 articles
Indexed 160/337 articles
Indexed 170/337 articles
Indexed 180/337 articles
Indexed 190/337 articles
Indexed 200/337 articles
Indexed 210/337 articles
Indexed 220/337 articles
Indexed 230/337 articles
Indexed 240/337 articles
Indexed 250/337 articles
Indexed 260/337 articles
Indexed 270/337 articles
Indexed 280/337 articles
Indexed 290/337 articles
Indexed 300/337 articles
Indexed 310/337 articles
Indexed 320/337 articles
Indexed 330/337 articles
{'answer': 'According to the provided context, the following drugs are used to treat inflammation:\n\n1. Corticosteroids (e.g., prednisone, hydrocortisone)\n2. 5

In [ ]:
# First configure the graphrag repo
# Edit graphrag/config.yaml with your credentials
import yaml

config = {
    'llm': {
        'provider': 'groq',
        'api_key': GROQ_API_KEY,
        'model': 'llama-3.3-70b-versatile'
    },
    'graph': {
        'host': TG_HOST,
        'username': TG_USERNAME,
        'password': TG_PASSWORD,
        'graph_name': 'MedicalKG'
    },
    'chunking': {
        'chunk_size': 256,
        'overlap': 30
    }
}
with open('graphrag/config.yaml', 'w') as f:
    yaml.dump(config, f)
print('Config written')

Config written


In [ ]:
import pyTigerGraph as tg

conn = tg.TigerGraphConnection(
    host="https://tg-bd330f89-6b25-42d4-a23e-96bacdee254d.tg-2635877100.i.tgcloud.io",
    graphname="pharmakg",           # or "MyGraph"
    username="pharmauser",
    password="cvRNWWfkBkQb4Eb4"
)
print(conn.echo())  # Must print "Hello TigerGraph"

Hello GSQL


In [ ]:
# Create the graph
conn.gsql('''
CREATE GRAPH MedicalKG()
USE GRAPH MedicalKG

CREATE SCHEMA_CHANGE JOB medical_schema FOR GRAPH MedicalKG {
  ADD VERTEX Entity(PRIMARY_ID entity_id STRING,
                    name STRING,
                    entity_type STRING,
                    description STRING)
    WITH primary_id_as_attribute="true";

  ADD VERTEX Document(PRIMARY_ID doc_id STRING,
                      title STRING,
                      content STRING)
    WITH primary_id_as_attribute="true";

  ADD UNDIRECTED EDGE RELATED_TO(FROM Entity, TO Entity,
                                  relationship_type STRING,
                                  weight FLOAT);

  ADD DIRECTED EDGE MENTIONED_IN(FROM Entity, TO Document);
}
RUN SCHEMA_CHANGE JOB medical_schema
''')
print("Schema created!")

Schema created!


In [ ]:
import shutil, os, json, pickle

# 1. Save the workspace files to Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create a backup folder
backup_path = "/content/drive/MyDrive/graphrag_hackathon_backup"
os.makedirs(backup_path, exist_ok=True)

# Copy all essential files
for fname in ["streamlit_app.py", "pipelines.py", "medical_dataset.jsonl",
              "indian_pharma_triples.json", "graph.pkl", "benchmark_report.csv"]:
    if os.path.exists(f"/content/{fname}"):
        shutil.copy(f"/content/{fname}", backup_path)
        print(f"✅ Saved {fname}")

# Also save the .streamlit config
if os.path.exists("/content/.streamlit"):
    shutil.copytree("/content/.streamlit", f"{backup_path}/.streamlit")

# Save API keys (if not in secrets, export them)
with open(f"{backup_path}/keys.json", "w") as f:
    json.dump({
        "GROQ": GROQ_API_KEY,
        "GEMINI": GEMINI_API_KEY,
        "TG_HOST": TG_HOST,
        "TG_PASSWORD": TG_PASSWORD  # keep in mind security
    }, f)

print("All files backed up to Google Drive.")

Mounted at /content/drive
✅ Saved streamlit_app.py
✅ Saved pipelines.py
✅ Saved graph.pkl
All files backed up to Google Drive.


In [ ]:
import networkx as nx
import chromadb
import time
import json
from groq import Groq
from sentence_transformers import SentenceTransformer

groq_client = Groq(api_key=GROQ_API_KEY)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# ── GRAPH (replaces TigerGraph for now) ──────────────────────
G = nx.DiGraph()

def build_graph(articles_list):
    for i, article in enumerate(articles_list):
        prompt = f"""Extract entities and relationships. Return ONLY JSON:
{{"entities":[{{"name":"X","type":"Drug"}}],
  "relationships":[{{"from":"X","to":"Y","type":"treats"}}]}}
Text: {article['text'][:600]}"""
        try:
            resp = groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role":"user","content":prompt}],
                max_tokens=500, temperature=0
            )
            raw = resp.choices[0].message.content
            if "```json" in raw:
                raw = raw.split("```json")[1].split("```")[0]
            elif "```" in raw:
                raw = raw.split("```")[1].split("```")[0]
            data = json.loads(raw.strip())
            for e in data.get("entities", []):
                G.add_node(e["name"],
                           type=e.get("type","Unknown"),
                           source=article["title"])
            for r in data.get("relationships", []):
                G.add_edge(r["from"], r["to"],
                           relation=r.get("type","related"))
        except:
            pass
        if i % 5 == 0:
            print(f"  Graph: {i+1}/{len(articles_list)} | "
                  f"nodes={G.number_of_nodes()} edges={G.number_of_edges()}")
        time.sleep(0.4)

print("Building knowledge graph...")
build_graph(articles[:30])
print(f"Graph ready: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building knowledge graph...
  Graph: 1/30 | nodes=11 edges=9
  Graph: 6/30 | nodes=49 edges=42
  Graph: 11/30 | nodes=84 edges=72
  Graph: 16/30 | nodes=121 edges=99
  Graph: 21/30 | nodes=155 edges=128
  Graph: 26/30 | nodes=186 edges=156
Graph ready: 218 nodes, 182 edges


In [ ]:
chroma = chromadb.Client()
collection = chroma.create_collection("medical")

def chunk(text, size=400):
    words = text.split()
    return [" ".join(words[i:i+size]) for i in range(0, len(words), size-50)]

print("Building vector index...")
for i, article in enumerate(articles[:30]):
    chunks = chunk(article["text"])
    for j, c in enumerate(chunks[:10]):   # max 10 chunks per article
        collection.add(documents=[c],
                       ids=[f"d{i}_c{j}"],
                       metadatas=[{"title": article["title"]}])
    if i % 5 == 0:
        print(f"  Indexed {i+1}/30 articles")

print("Vector index ready!")

Building vector index...
  Indexed 1/30 articles
  Indexed 6/30 articles
  Indexed 11/30 articles
  Indexed 16/30 articles
  Indexed 21/30 articles
  Indexed 26/30 articles
Vector index ready!


In [ ]:
from pyngrok import ngrok
import subprocess, time

subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(2)

subprocess.Popen([
    "streamlit", "run", "/content/streamlit_app.py",
    "--server.port=8501",
    "--server.headless=true"
])
time.sleep(5)

ngrok.kill()
tunnel = ngrok.connect(8501)
print("\n" + "="*50)
print("👉 OPEN THIS IN YOUR BROWSER:")
print(tunnel.public_url)
print("="*50)


👉 OPEN THIS IN YOUR BROWSER:
https://replica-enviable-outdated.ngrok-free.dev


In [ ]:
!streamlit run /content/streamlit_app.py --server.headless=true --server.port=8501 2>&1 | head -30

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: /content/streamlit_app.py


In [ ]:
import json, time, os, pickle, subprocess, socket
from pyngrok import ngrok

# ============================================================
# 1. SAVE YOUR GRAPH (if it's still in memory)
# ============================================================
# If you have a variable 'G' from earlier, we'll use it. If not, we'll build a small graph from the triples you generated.
# Let's check if 'all_triples' exists. If not, we'll create a tiny graph for testing.
try:
    # Try to load triples from file if saved earlier
    with open("/content/indian_pharma_triples.json", "r") as f:
        all_triples = json.load(f)
    print(f"Loaded {len(all_triples)} triples from file.")
except:
    # If no file, generate a small set of triples from scratch (this takes ~30 seconds)
    print("Generating fresh triples...")
    from groq import Groq
    groq_client = Groq(api_key="YOUR_GROQ_API_KEY")  # replace with your key or use the variable from earlier
    all_triples = []
    prompt = """Generate 10 JSON objects representing Indian pharmaceutical regulatory facts..."""
    # (abbreviated for brevity; use the triple-generation code from earlier)
    # For speed, I'll create a hardcoded fallback set:
    all_triples = [
        {"drug_name":"Cipla-Metformin","manufacturer":"Cipla","approved_for":["type 2 diabetes"],"contraindications":["kidney disease"],"interacts_with":["aspirin"],"approval_status":"Approved"},
        {"drug_name":"Sun-Atorvastatin","manufacturer":"Sun Pharma","approved_for":["high cholesterol"],"contraindications":["liver disease"],"interacts_with":["warfarin"],"approval_status":"Approved"},
        {"drug_name":"DrReddy-Omeprazole","manufacturer":"Dr. Reddy's","approved_for":["GERD"],"contraindications":[],"interacts_with":["clopidogrel"],"approval_status":"Approved"},
        {"drug_name":"Cipla-Aspirin","manufacturer":"Cipla","approved_for":["pain","inflammation"],"contraindications":["bleeding disorders"],"interacts_with":["ibuprofen"],"approval_status":"Approved"},
        {"drug_name":"Sun-Ibuprofen","manufacturer":"Sun Pharma","approved_for":["pain","inflammation"],"contraindications":["ulcers"],"interacts_with":["aspirin"],"approval_status":"Approved"},
        {"drug_name":"DrReddy-Metformin","manufacturer":"Dr. Reddy's","approved_for":["type 2 diabetes"],"contraindications":["kidney disease"],"interacts_with":["contrast dye"],"approval_status":"Approved"}
    ]
    print(f"Using {len(all_triples)} hardcoded triples.")

# Build NetworkX graph from triples
import networkx as nx
G = nx.Graph()
for t in all_triples:
    drug = t['drug_name']
    G.add_node(drug, type="Drug")
    G.add_node(t['manufacturer'], type="Company")
    G.add_edge(drug, t['manufacturer'], relation="manufactured_by")
    for dis in t['approved_for']:
        G.add_node(dis, type="Disease")
        G.add_edge(drug, dis, relation="treats")
    for dis in t['contraindications']:
        G.add_node(dis, type="Disease")
        G.add_edge(drug, dis, relation="contraindicated_for")
    for other in t['interacts_with']:
        G.add_node(other, type="Drug")
        G.add_edge(drug, other, relation="interacts_with")

# Save the graph
with open("/content/graph.pkl", "wb") as f:
    pickle.dump(G, f)
print(f"Graph saved with {len(G.nodes())} nodes and {len(G.edges())} edges.")

# ============================================================
# 2. WRITE THE PIPELINES MODULE
# ============================================================
pipelines_code = f'''
import pickle, time, json
import networkx as nx
import chromadb
from groq import Groq
from sentence_transformers import SentenceTransformer

GROQ_API_KEY = "{GROQ_API_KEY}"
groq_client = Groq(api_key=GROQ_API_KEY)

# Load the graph
with open("/content/graph.pkl", "rb") as f:
    G = pickle.load(f)

# Setup ChromaDB for Basic RAG (use the collection you already created, or create a new one)
chroma_client = chromadb.Client()
try:
    collection = chroma_client.get_collection("medical_rag")
except:
    collection = chroma_client.create_collection("medical_rag")
    # If collection is empty, you'll need to add documents before calling pipeline_basic_rag.
    # For now, we'll handle the case of empty collection.

def pipeline_llm_only(question):
    start = time.time()
    resp = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {{"role": "system", "content": "Answer concisely."}},
            {{"role": "user", "content": question}}
        ],
        max_tokens=400
    )
    latency = time.time() - start
    tokens = resp.usage.total_tokens
    cost = tokens * 0.00000059
    return {{
        "answer": resp.choices[0].message.content,
        "tokens": tokens,
        "latency": round(latency, 2),
        "cost": round(cost, 6),
        "path": []
    }}

def pipeline_basic_rag(question):
    start = time.time()
    # Try to query collection, if empty fallback to LLM-only
    try:
        results = collection.query(query_texts=[question], n_results=3)
        contexts = results["documents"][0]
        context = "\\n".join(contexts)
    except:
        context = "No indexed documents found. Answering without context."
    prompt = f"Answer using only this context:\\n{{context}}\\n\\nQuestion: {{question}}"
    resp = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {{"role": "system", "content": "You are a helpful assistant."}},
            {{"role": "user", "content": prompt}}
        ],
        max_tokens=400
    )
    latency = time.time() - start
    tokens = resp.usage.total_tokens
    cost = tokens * 0.00000059
    return {{
        "answer": resp.choices[0].message.content,
        "tokens": tokens,
        "latency": round(latency, 2),
        "cost": round(cost, 6),
        "path": []
    }}

def pipeline_graphrag(question):
    start = time.time()
    # Extract keywords and find seed nodes in graph
    keywords = [w.lower() for w in question.replace("?","").split() if len(w)>3]
    seeds = [n for n in G.nodes() if any(k in n.lower() for k in keywords)][:3]
    triples = []
    path = []
    for node in seeds:
        for nb in list(G.neighbors(node))[:4]:
            rel = G[node][nb].get("relation","related")
            triples.append(f"{{node}} {{rel}} {{nb}}")
            path.append(f"{{node}} →[{{rel}}]→ {{nb}}")
            # 2-hop
            for nb2 in list(G.neighbors(nb))[:2]:
                rel2 = G[nb][nb2].get("relation","related")
                triples.append(f"{{nb}} {{rel2}} {{nb2}}")
                path.append(f"{{nb}} →[{{rel2}}]→ {{nb2}}")
    context = "\\n".join(triples[:10]) if triples else "No relevant graph connections found."
    resp = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {{"role": "system", "content": "Answer based only on this graph knowledge:\\n" + context}},
            {{"role": "user", "content": question}}
        ],
        max_tokens=400
    )
    latency = time.time() - start
    tokens = resp.usage.total_tokens
    cost = tokens * 0.00000059
    return {{
        "answer": resp.choices[0].message.content,
        "tokens": tokens,
        "latency": round(latency, 2),
        "cost": round(cost, 6),
        "path": path[:5],
        "graph_context": context
    }}
'''

with open("/content/pipelines.py", "w") as f:
    f.write(pipelines_code)
print("pipelines.py written.")

# ============================================================
# 3. WRITE THE STREAMLIT APP
# ============================================================
app_code = '''
import streamlit as st
import plotly.graph_objects as go
import sys
sys.path.insert(0, '/content')
from pipelines import pipeline_llm_only, pipeline_basic_rag, pipeline_graphrag

st.set_page_config(page_title="GraphRAG Benchmark", layout="wide")
st.title("🧠 GraphRAG vs Basic RAG vs LLM-Only")
st.caption("Indian Pharma Knowledge Graph · Powered by Groq Llama 3.3 70B & TigerGraph")

# Sidebar ROI
st.sidebar.header("💰 Monthly Cost Projector")
qpd = st.sidebar.slider("Queries per day", 1000, 500000, 10000, step=1000)
monthly = qpd * 30

query = st.text_input(
    "Enter your medical question:",
    "What drugs treat inflammation in rheumatoid arthritis?"
)

if st.button("🚀 Run All 3 Pipelines", type="primary"):
    with st.spinner("Running all pipelines..."):
        r1 = pipeline_llm_only(query)
        r2 = pipeline_basic_rag(query)
        r3 = pipeline_graphrag(query)

    col1, col2, col3 = st.columns(3)

    with col1:
        st.subheader("Pipeline 1: LLM Only")
        st.error(r1["answer"])
        m1, m2, m3 = st.columns(3)
        m1.metric("Tokens", r1["tokens"])
        m2.metric("Latency", f"{r1['latency']}s")
        m3.metric("Cost", f"${r1['cost']:.5f}")

    with col2:
        st.subheader("Pipeline 2: Basic RAG")
        st.warning(r2["answer"])
        m1, m2, m3 = st.columns(3)
        delta2 = round((r2["tokens"]-r1["tokens"])/r1["tokens"]*100) if r1["tokens"] else 0
        m1.metric("Tokens", r2["tokens"], delta=f"{delta2}%")
        m2.metric("Latency", f"{r2['latency']}s")
        m3.metric("Cost", f"${r2['cost']:.5f}")

    with col3:
        st.subheader("Pipeline 3: GraphRAG ✨")
        st.success(r3["answer"])
        m1, m2, m3 = st.columns(3)
        delta3 = round((r3["tokens"]-r1["tokens"])/r1["tokens"]*100) if r1["tokens"] else 0
        m1.metric("Tokens", r3["tokens"], delta=f"{delta3}%")
        m2.metric("Latency", f"{r3['latency']}s")
        m3.metric("Cost", f"${r3['cost']:.5f}")

    # Graph reasoning path
    if r3.get("path"):
        st.subheader("🔗 GraphRAG Reasoning Path")
        path_html = " → ".join([
            f'<span style="background:#E1F5EE;padding:3px 8px;border-radius:12px;margin:2px;display:inline-block;font-size:13px">{p}</span>'
            for p in r3["path"]
        ])
        st.markdown(path_html, unsafe_allow_html=True)
        st.caption("Only GraphRAG can show this traversal path.")

    # Token bar chart
    st.subheader("📊 Token Usage Comparison")
    fig = go.Figure(go.Bar(
        x=["LLM Only", "Basic RAG", "GraphRAG"],
        y=[r1["tokens"], r2["tokens"], r3["tokens"]],
        marker_color=["#E24B4A", "#EF9F27", "#1D9E75"],
        text=[r1["tokens"], r2["tokens"], r3["tokens"]],
        textposition="outside"
    ))
    fig.update_layout(height=300, margin=dict(t=20,b=20), yaxis_title="Tokens used")
    st.plotly_chart(fig, use_container_width=True)

    # Savings callouts
    red_llm = round((r1["tokens"]-r3["tokens"])/r1["tokens"]*100) if r1["tokens"] else 0
    red_rag = round((r2["tokens"]-r3["tokens"])/r2["tokens"]*100) if r2["tokens"] else 0
    c1, c2 = st.columns(2)
    c1.success(f"GraphRAG uses {red_llm}% fewer tokens than LLM-Only")
    c2.info(f"GraphRAG uses {red_rag}% fewer tokens than Basic RAG")

    # ROI sidebar update
    st.sidebar.markdown("---")
    st.sidebar.metric("LLM Only / month",  f"${monthly * r1['cost']:,.0f}")
    st.sidebar.metric("Basic RAG / month", f"${monthly * r2['cost']:,.0f}")
    st.sidebar.metric("GraphRAG / month",  f"${monthly * r3['cost']:,.0f}",
                      delta=f"-${monthly * (r1['cost'] - r3['cost']):,.0f} saved")

    # Expandable graph context
    with st.expander("🔍 See raw graph context sent to LLM"):
        st.code(r3.get("graph_context", "none"), language="text")
'''

with open("/content/streamlit_app.py", "w") as f:
    f.write(app_code)
print("streamlit_app.py written.")

# ============================================================
# 4. LAUNCH STREAMLIT WITH PROPER WAIT
# ============================================================
# Kill any old Streamlit
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(2)

# Start Streamlit
proc = subprocess.Popen(
    ["streamlit", "run", "/content/streamlit_app.py",
     "--server.port=8501",
     "--server.headless=true",
     "--server.enableCORS=false",
     "--server.enableXsrfProtection=false"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait until port 8501 is accepting connections
print("Waiting for Streamlit to start...")
started = False
for i in range(30):
    time.sleep(1)
    try:
        s = socket.create_connection(("localhost", 8501), timeout=1)
        s.close()
        started = True
        print("Streamlit is ready!")
        break
    except:
        pass

if not started:
    # Print error logs
    print("Streamlit failed to start. STDERR:")
    print(proc.stderr.read().decode())
    print("STDOUT:")
    print(proc.stdout.read().decode())
else:
    # Kill old ngrok and open new tunnel
    ngrok.kill()
    time.sleep(1)
    tunnel = ngrok.connect(8501)
    print("\n" + "="*50)
    print("👉 OPEN THIS IN YOUR BROWSER:")
    print(tunnel.public_url)
    print("="*50)

Generating fresh triples...
Using 6 hardcoded triples.
Graph saved with 23 nodes and 25 edges.
pipelines.py written.
streamlit_app.py written.
Waiting for Streamlit to start...
Streamlit is ready!

👉 OPEN THIS IN YOUR BROWSER:
https://replica-enviable-outdated.ngrok-free.dev


In [ ]:
import os
os.makedirs("/content/.streamlit", exist_ok=True)

config_toml = '''
[theme]
# Professional Medical Blue — trusted, clinical, credible
primaryColor = "#0066cc"
backgroundColor = "#fafbfc"
secondaryBackgroundColor = "#f4f6f8"
textColor = "#2c3e50"
font = "sans serif"

[theme.sidebar]
backgroundColor = "#1a2332"
textColor = "#e8eaed"
secondaryBackgroundColor = "#243447"

[browser]
gatherUsageStats = false

[server]
enableCORS = false
enableXsrfProtection = false
maxUploadSize = 200
'''

with open("/content/.streamlit/config.toml", "w") as f:
    f.write(config_toml)
print("✅ Theme config written. Restart Streamlit to see changes.")

✅ Theme config written. Restart Streamlit to see changes.


In [ ]:
# Run ingestion (this takes 20-60 min depending on dataset size)
!python -m graphrag.ingest --config graphrag/config.yaml --input medical_dataset.jsonl

# After done, test a query
!python -m graphrag.query --config graphrag/config.yaml --query 'What drugs treat inflammation?'

/usr/bin/python3: No module named graphrag.ingest
/usr/bin/python3: No module named graphrag.query
